# Lecture 4: Portfolio Backtesting



## 4.0 Backtesting

A **backtest** is a historical simulation of a trading strategy: *if I had followed this rule in the past, what would have happened?* While investors are ultimately interested in net returns, a hedge fund might internally investigate a trading idea by first looking at the gross return in a backtest. Indeed, the hedge fund may first try to determine whether a strategy has any merit and, for this, examine whether the gross returns appear to be reliably positive. If the strategy appears to make money, the hedge fund will next consider whether the strategy survives **transaction costs** and, ultimately, whether it will add value for investors. While the fund's realized returns are naturally net of transaction costs, how to adjust a backtest for transaction costs is more involved.

Backtest should faithfully represent how markets operate, how trades are executed, what orders are available. There are also some implementation aspects that can bias the results of backtests and increase the risk of confusing in-sample performance with generalizable patterns that will also hold out of sample.

- **Look-ahead bias**: This potential flaw emerges when we develop a trading strategy **using historical data before it was actually known/available**. Some examples include corrections of reported financial statements after their publication, stock splits, or reverse splits.
- **Survivorship bias**: This bias arises when we **backtest only using data about securities that are currently active/tradeable**. By doing so, we omit the assets that have disappeared over time (due to bankruptcy, delisting, acquisition, and so on). Most of the time, those assets did not
perform well and our strategies can be skewed by failing to include those, as those assets could have been picked up in the past when they were still available in the markets
- **Outlier detection and treatment**: The main challenge is to discern the outliers that are not representative of the analyzed period as opposed to the ones that are an integral part of the market behavior.
- **Representative sample period**: As the goal of the backtest is to provide an indication of future performance, the sample data should reflect the current, and potentially also future, market behavior. By not spending enough time on this part, we can miss some crucial market regime aspects such as volatility (too few/many extreme events) or volume (too few data points).
- **Meeting investment objectives and constraints over time**: It can happen that a strategy leads to good performance at the very end of the evaluation period. However, in some periods when it was active, it resulted in unacceptably high losses or volatility. We could potentially track
those by using rolling performance/risk metrics, for example, the Sharpe ratio.
- **Realistic trading environment**: failing to include transaction costs can greatly impact the end result of a backtest. What is more, real-life trading involves further complications. For example, it might not be possible to execute all trades at all times or at the target price. Some of the things to consider are slippage (the difference between the expected price of a trade and the price at which the trade is executed), the availability of a counterparty for short positions, broker fees, commissions, bid-ask spreads, and so on.

**Discussion**: *A strategy rule says "buy when today's closing price is above the 20-day average of closing prices." What is the earliest moment you could actually place that trade?*

In [ ]:
# Setup
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf

---
## 4.1 A strategy that will make us rich?

Last week you learned to download prices and compute returns. Below is a complete backtest of a simple, popular trend-following rule on Apple stock:

- **Buy** (be long) whenever the closing price is above its 20-day simple moving average (SMA)
- **Sell** (hold nothing) whenever it is below
- No short selling

The code runs top to bottom. Nothing is hidden.

### Predict

Before running: a simple rule like this, on one mega-cap stock, published in every trading textbook — roughly what average annual return above buy-and-hold do you expect the backtest to show? **(a)** clearly negative **(b)** about zero **(c)** a few % **(d)** &gt;10% per year

In [ ]:
# --- The backtest -----------------------------------------------------
# download daily prices
data = yf.download('AAPL', start='2017-01-01', end='2025-01-01',
                   interval='1d', progress=False, auto_adjust=True)
df = data['Close'].copy()

# log returns and the 20-day SMA
df.loc[:, 'log_rtn'] = np.log(df['AAPL']).diff(1)
df.loc[:, 'sma_20'] = df['AAPL'].rolling(window=20).mean()
df.dropna(inplace=True) # modify the df

# position: long (1) when price is above the SMA, flat (0) otherwise
df.loc[:, 'position'] = 0
df.loc[df.AAPL > df.sma_20, 'position'] = 1

# strategy returns
df.loc[:, 'strategy_rtn'] = df['position'] * df['log_rtn']
df.loc[:, 'strategy_rtn_cum'] = df['strategy_rtn'].cumsum().apply(np.exp)
df.loc[:, 'bh_rtn_cum'] = df['log_rtn'].cumsum().apply(np.exp)

df[['bh_rtn_cum', 'strategy_rtn_cum']].plot(title='Cumulative returns: strategy vs. buy-and-hold')
plt.tight_layout()

n_years = len(df) / 252
print(f"Buy-and-hold: {df['bh_rtn_cum'].iloc[-1]:.2f}x over {n_years:.1f} years")
print(f"Strategy:     {df['strategy_rtn_cum'].iloc[-1]:.2f}x over {n_years:.1f} years")
print(f"Strategy annualized Sharpe: "
      f"{df['strategy_rtn'].mean() / df['strategy_rtn'].std() * np.sqrt(252):.2f}")

### Discussion

**Would you invest your client's money in this strategy?** (a) yes (b) no (c) I'd want to check something first — what?

The bug-free cross-examination starts with the *timing* of information:

1. On the day the price first crosses above the SMA — which day's closing price was needed to *know* that the crossing happened?
2. In the line `df['strategy_rtn'] = df['position'] * df['log_rtn']` — on that crossing day, does the strategy earn that day's return?
3. So: at what moment is this simulation implicitly *placing the trade*? Is that moment possible?

<details>
<summary><b>Solution </b></summary>

The signal on day $t$ needs day $t$'s closing price (the SMA condition compares the close to the average of closes). So the earliest you can act on it is at the close of day $t$, and even that assumes you can trade at the very price that generated the signal. The first return you can possibly earn is therefore day $t{+}1$'s close-to-close return.

But `df['position'] * df['log_rtn']` credits the strategy with day $t$'s return — the very return that *generated* the signal. The simulation is trading on information from the future: **look-ahead bias**.

Why does this flatter the results so much? Because on the day the price crosses *above* its average, the day's return is typically large and positive — the crossing and the up-move are the same event. The buggy backtest systematically harvests exactly those days.

The fix is one function call — `.shift(1)` — which lags the position by one day so that today's return is earned by *yesterday's* (implementable) position:

```python
df['strategy_rtn'] = df['position'].shift(1) * df['log_rtn']
```

</details>

In [ ]:
# --- The honest version: trade only on information you had ------------
df.loc[:, 'strategy_rtn'] = df['position'].shift(1) * df['log_rtn']   # <- the fix
df.loc[:, 'strategy_rtn_cum'] = df['strategy_rtn'].cumsum().apply(np.exp)

df[['bh_rtn_cum', 'strategy_rtn_cum']].plot(
    title='Cumulative returns, after fixing the look-ahead bias')
plt.tight_layout()

print(f"Buy-and-hold: {df['bh_rtn_cum'].iloc[-1]:.2f}x")
print(f"Strategy:     {df['strategy_rtn_cum'].iloc[-1]:.2f}x")
print(f"Strategy annualized Sharpe: "
      f"{df['strategy_rtn'].mean() / df['strategy_rtn'].std() * np.sqrt(252):.2f}")

### Takeaway

If we know that the position is *decided* on day $t$, which day's return does it earn? Three candidates:

- `lag = 0`: the position of day $t$ earns the return of day $t$. Impossible — the signal needs the close of $t$, which is also the price you would be "buying" at. This is **look-ahead bias**.
- `lag = 1`: assumes you trade at the closing price you just used for the signal (a market-on-close assumption, slightly optimistic).
- `lag = 2`: executed the day after. Fully safe.

---
## 4.2 Vectorized backtesting, step by step

We will backtest a very simple strategy with the following set of rules:
- We enter a long position if the close price is above the 20-day Simple Moving Average (SMA)
- We close the position when the close price goes below the 20-day SMA
- Short selling is not allowed
- The strategy is unit agnostic (we can enter a position of 1 share or 1000 shares) because we only care about the percentage change in the prices.

We backtest this strategy using Apple's stock and its historical prices from the years 2017 to 2024.

In [ ]:
# rebuild the dataset cleanly
data = yf.download('AAPL', start='2017-01-01', end='2025-01-01',
                   interval='1d', progress=False, auto_adjust=True)
df = data['Close'].copy()
df

In [ ]:
# compute log returns
df.loc[:, 'log_rtn'] = np.log(df['AAPL']).diff(1)
# compute 20-day simple moving averages of the prices
df.loc[:, 'sma_20'] = df['AAPL'].rolling(window=20).mean()
df.head(25)   # <- inspect: where do the NaNs end, and why there?

Create a position indicator:

In [ ]:
df.dropna(inplace=True)

# position indicator
df.loc[:,'position'] = 0
long_position = df.AAPL > df.sma_20
df.loc[long_position,'position'] = 1
df.describe()

### Explain

What does this expression compute? Answer in words *before* running:

```python
np.sum((df['position'] == 1) & (df['position'].shift(1) == 0))
```

In [ ]:
np.sum((df['position'] == 1) & (df['position'].shift(1) == 0))

<details>
<summary> <b>Solution</b></summary>

The number of times the strategy **enters** a long position: days where the position is 1 *and* the previous day's position was 0 — i.e., buy trades. (Roughly the same number of exits exist too — each entry is eventually followed by one. This number is about to matter enormously: every entry and exit is a trade, and trades cost money.)

</details>

Visualize what the strategy actually does — always plot your position against the price before you trust any statistics:

In [ ]:
fig, ax = plt.subplots(2, sharex=True)
df.loc['2024', ['AAPL', 'sma_20']].plot(ax=ax[0])
df.loc['2024', 'position'].plot(ax=ax[1])
ax[0].set_title('Preview of our strategy in 2024')
ax[0].legend(['AAPL Close', '20-day SMA'])
plt.tight_layout()

Calculate the strategy's daily and cumulative returns:

In [ ]:
# strategy and benchmark returns -- with the correct timing convention
df.loc[:, 'strategy_rtn'] = df['position'].shift(1) * df['log_rtn']
df.loc[:, 'strategy_rtn_cum'] = df['strategy_rtn'].cumsum().apply(np.exp)


Add the buy-and-hold strategy for comparison:

In [ ]:
df.loc[:, 'bh_rtn_cum'] = df['log_rtn'].cumsum().apply(np.exp)

# figure
df[['bh_rtn_cum', 'strategy_rtn_cum']].plot(title='Cumulative returns')
plt.tight_layout()

### Practice

Pick one:

1. Change the SMA window from 20 days to 100 days and rerun. Does slower trading help or hurt here? How did the *number of trades* change?
2. Make the rule symmetric: long when price &gt; SMA, **short** (position = -1) when price &lt; SMA. What one line changes? How does the equity curve change?



In [ ]:
# solution


<details>
<summary> <b>Solution</b></summary>

1. Replace `window=20` with `window=100` (and rerun everything below it — remember the NaN window grows to 100 rows). Fewer crossings → fewer trades.
```python
# solution (1)
data = yf.download('AAPL', start='2017-01-01', end='2025-01-01',
                   interval='1d', progress=False, auto_adjust=True)
df_v1 = data['Close'].copy()
df_v1.loc[:, 'log_rtn'] = np.log(df_v1['AAPL']).diff(1)
# compute 20-day simple moving averages of the prices
df_v1.loc[:, 'sma_20'] = df_v1['AAPL'].rolling(window=100).mean()

# position indicator
df_v1.loc[:,'position'] = 0
long_position = df_v1.AAPL > df_v1.sma_20
df_v1.loc[long_position,'position'] = 1
df_v1.loc[:, 'strategy_rtn'] = df_v1['position'].shift(1) * df_v1['log_rtn']
df_v1.loc[:, 'strategy_rtn_cum'] = df_v1['strategy_rtn'].cumsum().apply(np.exp)
df_v1.loc[:, 'bh_rtn_cum'] = df_v1['log_rtn'].cumsum().apply(np.exp)

# figure
df_v1[['bh_rtn_cum', 'strategy_rtn_cum']].plot(title='Cumulative returns')
plt.tight_layout()
```
2. Only the position assignment changes:
```python
# solution (2)
data = yf.download('AAPL', start='2017-01-01', end='2025-01-01',
                   interval='1d', progress=False, auto_adjust=True)
df_v2 = data['Close'].copy()
df_v2.loc[:, 'log_rtn'] = np.log(df_v2['AAPL']).diff(1)
# compute 20-day simple moving averages of the prices
df_v2.loc[:, 'sma_20'] = df_v2['AAPL'].rolling(window=20).mean()

# position indicator
df_v2.loc[:,'position'] = 0
long_position = df_v2.AAPL > df_v2.sma_20
df_v2.loc[long_position,'position'] = 1
short_position = df_v2.AAPL < df_v2.sma_20 # or sma_100 if you renamed it
df_v2.loc[short_position,'position'] = -1

df_v2.loc[:, 'strategy_rtn'] = df_v2['position'].shift(1) * df_v2['log_rtn']
df_v2.loc[:, 'strategy_rtn_cum'] = df_v2['strategy_rtn'].cumsum().apply(np.exp)
df_v2.loc[:, 'bh_rtn_cum'] = df_v2['log_rtn'].cumsum().apply(np.exp)

# figure
df_v2[['bh_rtn_cum', 'strategy_rtn_cum']].plot(title='Cumulative returns')
plt.tight_layout()
```


The long-short version is *always* in the market, so it also loses money in downtrends when the signal is wrong — inspect 2022.

</details>

---
## 4.3 Transaction costs

Our strategy trades more than 150 times. Each trade pays the **bid-ask spread**, possibly a commission, and — for large investors — **price impact** (your own buying moves the price). A backtest that ignores this compares a *gross* fantasy to a *net* reality.

We model a proportional cost: each change of position costs `transaction_cost` × |change in position|.

### Predict

At 0.5% per position change (a reasonably conservative all-in figure for a retail investor in a liquid stock), does our SMA strategy still beat buy-and-hold? (a) easily (b) barely (c) it now loses

In [ ]:
transaction_cost = 0.005
df.loc[:, 'tc'] = df['position'].diff(1).abs() * transaction_cost
df.loc[:, 'strategy_rtn_cum_tc'] = (df['strategy_rtn'] - df['tc']).cumsum().apply(np.exp)

df[['bh_rtn_cum', 'strategy_rtn_cum', 'strategy_rtn_cum_tc']].plot(
    title='Cumulative returns, gross and net of costs')
plt.tight_layout()

print(f"Gross: {df['strategy_rtn_cum'].iloc[-1]:.2f}x   "
      f"Net:   {df['strategy_rtn_cum_tc'].iloc[-1]:.2f}x   "
      f"Buy-and-hold: {df['bh_rtn_cum'].iloc[-1]:.2f}x")

### Explore: the breakeven cost

Find (roughly) the transaction cost at which the strategy's final cumulative return equals buy-and-hold's. Approach: wrap the cost calculation in a loop over candidate costs, e.g. `np.arange(0, 0.0015, 0.0001)`, and print the final ratio for each.

Then discuss: is your breakeven above or below what a retail investor actually pays today? What about a fund trading €500m?





In [ ]:
# solution


<details>
<summary> <b>Solution</b></summary>

```python
for c in np.arange(0, 0.0015, 0.0001):
    tc = df['position'].diff(1).abs() * c
    final = (df['strategy_rtn'] - tc).cumsum().apply(np.exp).iloc[-1]
    print(f"cost {c:.5f}: strategy {final:.2f}x vs buy-and-hold "
          f"{df['bh_rtn_cum'].iloc[-1]:.2f}x")
```

The exact breakeven depends on the sample, but the lesson does not: **turnover is a tax on your signal**. When you evaluate any strategy — including your group assignment's — report turnover and show the net-of-cost result. A prospectus that only shows gross returns is not investable.

</details>

### A hidden assumption worth seeing

A classic "60/40" portfolio: 60% equities (SPY), 40% short-term bills (TBIL), combined as `0.6*r_SPY + 0.4*r_TBIL` every day.

**Discuss:** does that formula describe a portfolio you buy once and hold? Or does it quietly assume you *trade every day*?

In [ ]:
data = yf.download(['SPY', 'TBIL'], start='2022-12-31', end='2024-09-30',
                   interval='1d', progress=False, auto_adjust=True)
df6040 = data['Close'].apply(np.log).diff(1).dropna()

w = 0.6
ptf_60_40 = w * df6040['SPY'] + (1 - w) * df6040['TBIL']
ptf_60_40.cumsum().apply(np.exp).plot(title='60/40 portfolio, cumulative return')
plt.tight_layout()

<details>
<summary><b>Solution</b></summary>

Holding the weights *fixed at 60/40 every single day* means **rebalancing daily**: whenever stocks outperform, a buy-and-hold portfolio would drift above 60% equities, so keeping exactly 60% requires selling the winner every day. The innocent-looking formula smuggles in a daily-rebalanced, zero-cost trading strategy.
</details>

---
## 4.4 The AI wrote it — should you trust it?

A student asked an AI assistant: *"Write me a Python backtest of a momentum strategy: each month, hold the stock with the best return over the previous 12 months, out of AAPL, NVDA, MSFT, AMZN. Report the annualized Sharpe ratio."*

Below is the (lightly shortened) answer. It runs without errors and prints a spectacular Sharpe ratio. **There are three bugs and one deeper problem with the question itself.** Find as many as you can *before* opening the solution. Rank them: which one inflates the result most?

In [ ]:
# --- AI-generated code: DO NOT TRUST -- find the bugs -----------------
tickers = ['AAPL', 'NVDA', 'MSFT', 'AMZN']
data = yf.download(tickers, start='2015-01-01', end='2025-01-01',
                   interval='1mo', progress=False, auto_adjust=True)
prices = data['Close']
rets = prices.pct_change()

# momentum signal: trailing 12-month return
momentum = prices.pct_change(12).dropna()

# each month, hold the stock with the highest momentum
best = momentum.idxmax(axis=1)
strat_rets = pd.Series([rets.loc[m, best[m]] for m in best.index],
                       index=best.index)

# performance
cum = (1 + strat_rets).cumprod()
sharpe = strat_rets.mean() / strat_rets.std() * 12
print(f"Final value of $1: {cum.iloc[-1]:.2f}")
print(f"Annualized Sharpe ratio: {sharpe:.2f}")
cum.plot(title='Momentum strategy (AI version)')
plt.tight_layout()

<details>
<summary> <b>Solution — the three bugs and the deeper problem</b></summary>

**Bug 1 — look-ahead bias (the big one).** `momentum.idxmax(axis=1)` picks the winner using the 12-month return *ending this month* — which includes this month's return — and then `rets.loc[m, best[m]]` earns *this same month's* return. The strategy "buys" the stock after seeing that it just won. The winner's selection and its reward overlap by one month. Fix: lag the signal — `best = momentum.shift(1).idxmax(axis=1)` (and think carefully about whether one month of lag is even enough for information you would truly have had).

**Bug 2 — wrong annualization.** Sharpe ratios scale with the *square root* of time: monthly → annual multiplies by $\sqrt{12} \approx 3.46$, not 12. The printed Sharpe is inflated by a factor of $12/\sqrt{12} = \sqrt{12}$.

**Bug 3 — the Sharpe ratio uses raw returns, not excess returns.** The Sharpe ratio is *excess* return per unit of risk: $(\bar{r} - r_f)/\sigma$. The code never subtracts the risk-free rate, which flatters every strategy — especially in a high-rate environment. (Small habit worth forming: whenever you see `mean()/std()` presented as a Sharpe ratio, ask where $r_f$ went.)

**The deeper problem — survivorship in the question itself.** "AAPL, NVDA, MSFT, AMZN" is a universe chosen *in 2026, because these stocks won*. A momentum strategy on hand-picked winners is contaminated before a line of code runs. The AI answered the question as asked; the question already contained hindsight. **This is the part no assistant will fix for you.**

</details>


---
## 4.5 Practice: the SMA crossover strategy

The SMA crossover uses two moving averages: **buy** when the 50-day SMA crosses above the 200-day SMA ("golden cross"), **sell** when it crosses below ("death cross"). Build the full, honest backtest yourself — every step is one you have now done at least once.

1. Download daily prices of NVIDIA (NVDA), 2017-2024. Inspect the first and last rows.
2. Compute log returns, the 50-day SMA, and the 200-day SMA.
3. Create the position indicator (long when SMA50 &gt; SMA200, else flat).
4. Visualize price, both SMAs, and the position over 2023.
5. Compute strategy and buy-and-hold cumulative returns — **respecting the timing convention**.
6. Plot both. Then add transaction costs at 0.5% and plot again.
7. Evaluate performance with the metrics from Lecture 3 (or the `quantstats` package: `!pip -q install quantstats`).
8. **The cross-examination (this is the part that matters):** write, in words, three ways this backtest could still be lying to you — even with the timing convention respected.

<details>
<summary> <b>Solution — steps 1-6</b></summary>

```python
data = yf.download('NVDA', start='2017-01-01', end='2025-01-01',
                   interval='1d', progress=False, auto_adjust=True)
df = data['Close'].copy()

df['log_rtn'] = np.log(df['NVDA']).diff(1)
df['sma_50'] = df['NVDA'].rolling(window=50).mean()
df['sma_200'] = df['NVDA'].rolling(window=200).mean()
df.dropna(inplace=True)

df['position'] = 0
df.loc[df.sma_50 > df.sma_200, 'position'] = 1

fig, ax = plt.subplots(2, sharex=True)
df.loc['2020':'2023', ['NVDA', 'sma_50', 'sma_200']].plot(ax=ax[0])
df.loc['2020':'2023', 'position'].plot(ax=ax[1])
ax[0].set_title('Preview of the strategy in 2020-2023')
plt.tight_layout()

df['strategy_rtn'] = df['position'].shift(1) * df['log_rtn']
df['strategy_rtn_cum'] = df['strategy_rtn'].cumsum().apply(np.exp)
df['bh_rtn_cum'] = df['log_rtn'].cumsum().apply(np.exp)

tc = df['position'].diff(1).abs() * 0.005
df['strategy_rtn_cum_tc'] = (df['strategy_rtn'] - tc).cumsum().apply(np.exp)

df[['bh_rtn_cum', 'strategy_rtn_cum', 'strategy_rtn_cum_tc']].plot(
    title='Cumulative returns')
plt.tight_layout()
```

</details>

<details>
<summary> <b>Solution — step 8 (do not open until you have written your own three)</b></summary>

Among the good answers: **(i)** single-asset, single-period evidence — NVDA 2017-2024 is one extraordinary bull run; the same rule on another stock or decade may fail (unrepresentative sample); **(ii)** choosing NVDA *because* it went up is survivorship/selection bias in the research design; **(iii)** the 50/200 parameters are the most famous in technical analysis precisely because they were *searched for* by thousands of people on this same historical data — collective data-snooping; **(iv)** execution realism: crossings are detected at the close, fills at the next open can gap far from the backtest price for a volatile stock.

</details>

